# SPMamba HLS-CMDS - train tach tim/phoi tren Colab T4

Chuan bi truoc khi chay:
- Chon runtime GPU T4.
- Tao Colab Secret ten `WANDB_API_KEY` va bat `Notebook access`.
- Build `data_hls.h5` tren laptop, nen thanh zip, roi upload zip len Drive.


In [ ]:
# === Cell 1: Cau hinh duong dan + kiem tra GPU ===
from google.colab import drive
from pathlib import Path
import os, subprocess

drive.mount('/content/drive')

REPO_URL = "https://github.com/hoanganh0106/spmamba.git"
REPO_DIR = "/content/spmamba"
ENV_DIR  = "/content/spmamba-env"
MM       = "/content/bin/micromamba"

DRIVE_H5_ZIP   = "/content/drive/MyDrive/data_hls.zip"  # zip chua data_hls.h5
DRIVE_H5_PATH  = ""  # optional: /content/drive/MyDrive/data_hls.h5 neu upload thang file H5
DATA_WORK_DIR  = "/content/hls_h5"
LOCAL_H5       = "/content/data_hls.h5"
CONFIG_PATH    = f"{REPO_DIR}/configs/spmamba-hls-t4.yml"
CKPT_DIR       = "/content/drive/MyDrive/SPMamba_checkpoints/SPMamba-HLS-HeartLung-T4"

WANDB_PROJECT  = "spmamba"
WANDB_RUN_NAME = "SPMamba-HLS-HeartLung-T4"

WANDB_API_KEY = ""
try:
    from google.colab import userdata
    WANDB_API_KEY = userdata.get("WANDB_API_KEY") or ""
except Exception as e:
    print("[wandb] Khong doc duoc Colab Secret:", e)
WANDB_API_KEY = WANDB_API_KEY or os.environ.get("WANDB_API_KEY", "")
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    print("[wandb] Da co WANDB_API_KEY")
else:
    print("[wandb] Chua co WANDB_API_KEY. Tao secret va bat Notebook access roi chay lai cell nay.")

Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)
for k in ["REPO_DIR", "ENV_DIR", "DRIVE_H5_ZIP", "DRIVE_H5_PATH", "LOCAL_H5", "CONFIG_PATH", "CKPT_DIR"]:
    print(f"{k} = {globals()[k]}")
subprocess.run("nvidia-smi", shell=True, check=False)


In [ ]:
# === Cell 2: Cai moi truong + thu vien + dang nhap W&B ===
import os, subprocess, shutil
from pathlib import Path

os.environ["MAMBA_ROOT_PREFIX"] = "/content/micromamba-root"
os.environ["PATH"] = "/content/bin:" + os.environ.get("PATH", "")

INSTALL_DEPS = True
FORCE_RECREATE_ENV = False

def run(cmd, cwd=None, check=True):
    print(f"\n[cmd] {cmd}", flush=True)
    return subprocess.run(cmd, shell=True, cwd=cwd, env=os.environ.copy(), text=True, check=check)

Path('/content/bin').mkdir(parents=True, exist_ok=True)
if not Path(MM).exists():
    run("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content bin/micromamba")
run(f"{MM} --version")

if FORCE_RECREATE_ENV and Path(ENV_DIR).exists():
    shutil.rmtree(ENV_DIR)
if not Path(ENV_DIR).exists():
    run(f"{MM} create -p {ENV_DIR} python=3.10 --yes -q")

if INSTALL_DEPS:
    run(f"{MM} run -p {ENV_DIR} python -m pip install -U pip -q")
    run(f"{MM} run -p {ENV_DIR} pip install 'setuptools<70' typeguard==2.13.3 h5py pytorch-lightning==2.0.2 einops soundfile librosa fast-bss-eval speechbrain==0.5.14 hydra-core rich torch-complex packaging wandb -q")
    run(f"{MM} run -p {ENV_DIR} pip install --force-reinstall scipy -q")
    run(f"{MM} run -p {ENV_DIR} pip install torch-mir-eval torch-optimizer transformers -q")
    run(f"{MM} run -p {ENV_DIR} pip install torch==2.5.0 torchaudio==2.5.0 torchvision==0.20.0 --index-url https://download.pytorch.org/whl/cu124 --force-reinstall -q")
    run(f"{MM} run -p {ENV_DIR} pip install 'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.0/causal_conv1d-1.6.0+cu12torch2.5cxx11abiFALSE-cp310-cp310-linux_x86_64.whl' 'https://github.com/state-spaces/mamba/releases/download/v2.3.0/mamba_ssm-2.3.0+cu12torch2.5cxx11abiFALSE-cp310-cp310-linux_x86_64.whl' --no-deps --force-reinstall -q")

run(f"{MM} run -p {ENV_DIR} python -c \"import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())\"")
run(f"{MM} run -p {ENV_DIR} python -c \"from mamba_ssm.modules.mamba_simple import Mamba; print('mamba_ssm OK')\"")
run(f"{MM} run -p {ENV_DIR} python -c \"import wandb; print('wandb', wandb.__version__)\"")
run(f"{MM} run -p {ENV_DIR} python -c \"import os, wandb; k=os.environ.get('WANDB_API_KEY'); print('wandb login:', wandb.login(key=k, relogin=True)) if k else print('CHUA co WANDB_API_KEY')\"")
run("nvidia-smi", check=False)


In [ ]:
# === Cell 3: Clone/cap nhat repo ===
from pathlib import Path
import subprocess, os

def run_list(cmd, cwd=None, check=True):
    print("\n[cmd]", " ".join(cmd), flush=True)
    return subprocess.run(cmd, cwd=cwd, env=os.environ.copy(), text=True, check=check)

repo = Path(REPO_DIR)
if repo.exists():
    run_list(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
else:
    run_list(["git", "clone", REPO_URL, REPO_DIR])
run_list(["git", "-C", REPO_DIR, "status", "--short"], check=False)


In [ ]:
# === Cell 4: Giai nen/copy H5 da build san ve SSD local ===
from pathlib import Path
import h5py, shutil, zipfile

REFRESH_LOCAL_H5 = False
work_dir = Path(DATA_WORK_DIR)
work_dir.mkdir(parents=True, exist_ok=True)

if REFRESH_LOCAL_H5 and Path(LOCAL_H5).exists():
    Path(LOCAL_H5).unlink()

if not Path(LOCAL_H5).exists():
    if DRIVE_H5_PATH and Path(DRIVE_H5_PATH).exists():
        print("[data] copy H5 tu Drive:", DRIVE_H5_PATH)
        shutil.copy2(DRIVE_H5_PATH, LOCAL_H5)
    elif DRIVE_H5_ZIP and Path(DRIVE_H5_ZIP).exists():
        print("[data] giai nen H5 tu zip:", DRIVE_H5_ZIP)
        with zipfile.ZipFile(DRIVE_H5_ZIP) as zf:
            h5_members = [name for name in zf.namelist() if name.lower().endswith((".h5", ".hdf5")) and not name.endswith("/")]
            if not h5_members:
                raise RuntimeError(f"Zip khong co file .h5/.hdf5: {DRIVE_H5_ZIP}")
            if len(h5_members) > 1:
                print("[data] zip co nhieu H5, dung file dau tien:", h5_members)
            member = h5_members[0]
            extracted = work_dir / Path(member).name
            with zf.open(member) as src, open(extracted, "wb") as dst:
                shutil.copyfileobj(src, dst)
        shutil.copy2(extracted, LOCAL_H5)
        print("[data] extracted:", extracted)
    else:
        raise FileNotFoundError(f"Khong thay H5 zip/file tren Drive. Sua DRIVE_H5_ZIP hoac DRIVE_H5_PATH. DRIVE_H5_ZIP={DRIVE_H5_ZIP}")
else:
    print("[data] da co", LOCAL_H5)

with h5py.File(LOCAL_H5, "r") as f:
    print("[h5 attrs]", dict(f.attrs))
    first_id = f["mixture_ids"][0].decode("utf-8") if isinstance(f["mixture_ids"][0], bytes) else f["mixture_ids"][0]
    g = f["mixtures"][first_id]
    print("[h5 first]", first_id, g["mix"].shape, g["s1"].shape, g["s2"].shape, "has_s3=", "s3" in g)


In [ ]:
# === Cell 5: Train + W&B + resume bridge ===
import os, subprocess, shutil
from pathlib import Path

if not WANDB_API_KEY:
    raise RuntimeError("Chua co WANDB_API_KEY. Chay lai Cell 1 sau khi tao Colab Secret va bat Notebook access.")
if not Path(LOCAL_H5).exists():
    raise RuntimeError("Chua co /content/data_hls.h5. Chay Cell 4 truoc.")

os.chdir(REPO_DIR)
ckpt_root = Path(CKPT_DIR)
ckpt_root.mkdir(parents=True, exist_ok=True)
last_path = ckpt_root / "last.ckpt"
if not last_path.exists():
    others = [p for p in ckpt_root.glob("*.ckpt") if not p.name.startswith("last")]
    if others:
        newest = max(others, key=lambda p: p.stat().st_mtime)
        shutil.copy2(newest, last_path)
        print("[resume] tao last.ckpt tu", newest.name)
resume = last_path.exists()

cmd = [MM, "run", "-p", ENV_DIR, "python", "-u", "audio_train.py",
       "--conf_dir", CONFIG_PATH, "--checkpoint_dir", CKPT_DIR,
       "--wandb", "--wandb_project", WANDB_PROJECT, "--wandb_run_name", WANDB_RUN_NAME]
if resume:
    cmd.append("--resume")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONPATH"] = REPO_DIR
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["WANDB_API_KEY"] = WANDB_API_KEY

print("[train] lenh:", " ".join(cmd))
print("[train] resume:", resume)
proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    print(f"\n[train] ket thuc - exit code {proc.returncode}")
except KeyboardInterrupt:
    proc.terminate()
    print("\n[train] da dung (KeyboardInterrupt)")


In [ ]:
# === Cell 6: Liet ke checkpoint da luu ===
from pathlib import Path
import time

ckpt_root = Path(CKPT_DIR)
print("CKPT_DIR:", ckpt_root)
if ckpt_root.exists():
    for p in sorted(ckpt_root.glob("*"), key=lambda x: x.stat().st_mtime, reverse=True):
        size_mb = p.stat().st_size / (1024 * 1024)
        mtime = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(p.stat().st_mtime))
        print(f"{mtime}  {size_mb:8.1f} MiB  {p.name}")
else:
    print("Chua co checkpoint dir")
